# 03 - Dataset, Patch and Training Analysis

This notebook summarizes the prepared CNN patch dataset and the first lightweight CNN training run.

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "labels.csv"
PATCH_LABELS_PATH = PROJECT_ROOT / "data" / "processed" / "patch_labels.csv"
DATASET_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_summary.json"
TRAINING_DIR = PROJECT_ROOT / "outputs" / "training"

labels = pd.read_csv(LABELS_PATH)
patch_labels = pd.read_csv(PATCH_LABELS_PATH)
patch_labels.head()

## Patch Dataset Summary

In [ ]:
with open(DATASET_SUMMARY_PATH, "r", encoding="utf-8") as f:
    dataset_summary = json.load(f)

dataset_summary

## Class Counts By Split

In [ ]:
split_counts = patch_labels.groupby(["split", "label"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", figsize=(8, 4), title="Patch Counts By Split")
plt.xlabel("Split")
plt.ylabel("Patch count")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
split_counts

## Class Counts By Original Scenario

In [ ]:
scenario_counts = patch_labels.groupby(["scenario_type", "label"]).size().unstack(fill_value=0)
scenario_counts.plot(kind="bar", stacked=True, figsize=(11, 5), title="Patch Classes By Scenario")
plt.xlabel("Scenario")
plt.ylabel("Patch count")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", alpha=0.25)
scenario_counts

## Preview Correct and False Patches

In [ ]:
preview_paths = [
    PROJECT_ROOT / "outputs" / "patch-preview" / "correct_grid.png",
    PROJECT_ROOT / "outputs" / "patch-preview" / "false_grid.png",
]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, path in zip(axes, preview_paths):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(path.name)
    ax.axis("off")
plt.tight_layout()

## Training History

In [ ]:
history_path = TRAINING_DIR / "history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
    axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
else:
    print("Training history not found yet. Run src/train_classifier.py first.")

## Test Metrics and Training Artifacts

In [ ]:
metrics_path = TRAINING_DIR / "test_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        test_metrics = json.load(f)
    display(test_metrics)
else:
    print("Test metrics not found yet.")

for artifact in ["training_curves.png", "confusion_matrix.png", "sample_predictions.png"]:
    print(TRAINING_DIR / artifact)

## Notes

- Current data is synthetic only.
- False beacons are intentionally similar to true beacons, so top-1 mistakes are expected.
- The model should be re-tested on Unity frames before being used for closed-loop tracking.

## Phase 6 - Single-Frame Pipeline

Phase 6 connects preprocessing, bright-candidate detection, CNN patch classification and fused candidate ranking for one independent camera frame.

It writes a PID-ready JSON result and an annotated diagnostic image under `outputs/pipeline-test/`.


In [ ]:
import json
from pathlib import Path

pipeline_result_path = Path("../outputs/pipeline-test/result.json")
pipeline_image_path = Path("../outputs/pipeline-test/annotated_result.png")

if pipeline_result_path.exists():
    result = json.loads(pipeline_result_path.read_text(encoding="utf-8"))
    summary = {
        "target_found": result.get("target_found"),
        "status": result.get("status"),
        "candidate_count": result.get("candidate_count"),
        "selected_candidate_id": result.get("selected_candidate_id"),
        "cnn_probability": result.get("cnn_probability"),
        "cv_baseline_score": result.get("cv_baseline_score"),
        "fused_score": result.get("fused_score"),
        "control_error_x": result.get("control_error_x"),
        "control_error_y": result.get("control_error_y"),
    }
    summary
else:
    print("Run src/pipeline.py first to create outputs/pipeline-test/result.json")


In [ ]:
from IPython.display import Image, display

if pipeline_image_path.exists():
    display(Image(filename=str(pipeline_image_path)))
else:
    print("Run src/pipeline.py first to create outputs/pipeline-test/annotated_result.png")


## Phase 7 - Temporal Verification And Tracking

Phase 7 adds ordered-frame memory on top of the Phase 6 single-frame pipeline. The tracker uses candidate association, persistence-based temporal verification, and a constant-velocity Kalman filter to produce stable coordinates and lock-state outputs.

Generated smoke-test artifacts are expected under `outputs/tracking-test/`:

- `tracking_results.csv`
- `tracking_summary.json`
- `annotated_tracking.mp4`


In [ ]:
from pathlib import Path
import json
import pandas as pd

tracking_dir = Path("../outputs/tracking-test")
summary_path = tracking_dir / "tracking_summary.json"
results_path = tracking_dir / "tracking_results.csv"

print("summary exists:", summary_path.exists())
print("results exists:", results_path.exists())
print("video exists:", (tracking_dir / "annotated_tracking.mp4").exists())


In [ ]:
if summary_path.exists():
    tracking_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    tracking_summary
else:
    print("Run src/tracker.py first to generate the Phase 7 summary.")


In [ ]:
if results_path.exists():
    tracking_results = pd.read_csv(results_path)
    display(tracking_results[[
        "frame_index",
        "lock_state",
        "measurement_available",
        "using_prediction_only",
        "missed_frames",
        "target_found",
        "filtered_x_px",
        "filtered_y_px",
        "predicted_x_px",
        "predicted_y_px",
    ]].head(18))
else:
    print("Run src/tracker.py first to generate tracking_results.csv.")


### Phase 7 Notes

The smoke sequence intentionally hides the target for two frames. Healthy behavior is `LOCKED -> COASTING -> LOCKED`: the tracker should use prediction-only output during the short dropout and return to measured tracking once the beacon appears again.

Current smoke-test result:

```text
Total frames: 18
Measurements: 16
Locked frames: 14
Coasting frames: 2
Lost frames: 0
Maximum consecutive missed frames: 2
Mean association distance: 1.109044 px
```

No real tracking accuracy is claimed from this smoke test because it does not yet use ground-truth trajectory evaluation.
